# ABE1 recovery run — retrain the DeBERTa-v3-large bi-encoder on TR only
### One change of data, no change of recipe
- **data** — train 46,729 (TR: the ids ABE2 · TB1 · TB2 · Arith trained on) · valid 5,000 for selection · holdout 5,748 untouched
- **recipe** — V3 as it was: 3-block head, zero-init pooling, LR 1e-5, 3-epoch schedule, fp16, batch 16, seed 42
- **new** — quarter-epoch evals with EarlyStopping and a top-3 keep, length-sorted eval, id-sum checks, a packed deliverable

### 왜 다시 돌리는가
배포 중인 ABE1(V3 · checkpoint-3109)은 **49,729행**으로 학습했다. 그중 **3,000행**이 나중에 만든 valid 5,000 안에 들어가 있어서
최종 파이프라인은 그 3,000행을 결합기(stage2)에서 빼야 했다. 그대로 넣으면 holdout 이 +0.0043 나빠진다는 것이 측정돼 있다
(ABE1 오염으로 `t_abe1` 기여가 2.5배 부푼다).<br>
<br>
이 런은 **같은 레시피로 TR 46,729행만** 학습한다. 3,000행은 valid(체크포인트 선택)로만 보고 그래디언트는 한 번도 받지 않는다.
끝나면 그 3,000행이 다섯 전문가 모두에게 깨끗해지고 stage2 가 **7,748 → 10,748행**이 된다.<br>
<br>
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">분할</th><th style="text-align:right">행</th><th style="text-align:left">이 런에서</th><th style="text-align:left">회수 후</th></tr></thead>
<tbody>
<tr><td>train</td><td style="text-align:right">46,729</td><td>그래디언트</td><td>층1 학습 (그대로)</td></tr>
<tr><td>valid</td><td style="text-align:right">5,000</td><td>체크포인트 선택만 (clean 2,000 + <b>회수 3,000</b>)</td><td>stage2</td></tr>
<tr><td>holdout</td><td style="text-align:right">5,748</td><td>보고만</td><td>stage2 (채점)</td></tr>
</tbody></table>
</div><br>
레시피는 V3 그대로이고, **바꾼 것은 데이터와 평가 주기뿐**이다.

# 1. Import / 환경

### Parameters — papermill overrides these with `-p` / 실행 노브
papermill 이 `QUICK`·경로를 넘긴다. A40 이 아닌 카드에서 최악배치 프로브가 OOM 이면 `BATCH_SIZE 8 · GRADIENT_ACCUMULATION_STEPS 2` 로
내린다 — 유효 배치 16 은 5절이 assert 로 지킨다.

In [1]:
# Run-control knobs — papermill overrides them with -p. Everything that shapes the model lives in section 5.
# 실행 제어 노브 — papermill 이 -p 로 덮어쓴다. 모델을 정하는 값은 5절 하이퍼파라미터 셀에만 있다.
# (no "=" inside the comments below: papermill skips any parameter line holding more than one)
QUICK = False                                     # True for a wiring run on a slice, about 10 min on an A40
PKG_DIR = '/workspace/ABE1'                       # where this package was unpacked / 이 패키지를 푼 곳
OUT_DIR = '/workspace/abe1_out/full'              # outputs, on the persistent volume / 산출물 — 영구 볼륨 위 절대경로
BACKBONE_DIR = '/workspace/hf/deberta-v3-large'   # pretrained weights — cell 14 fetches the pinned commit / 14번 셀이 고정 커밋을 받는다
BATCH_SIZE = 16                                   # A40 48GB runs 16 x 1 as V3 did. On OOM: 8 x 2, then 4 x 4
GRADIENT_ACCUMULATION_STEPS = 1                   # effective batch must stay 16 (section 5 asserts it)
EVAL_BATCH_SIZE = 32                              # V3 value. The eval-path probe in section 6 checks its worst batch

In [2]:
# Parameters
QUICK = False
PKG_DIR = "/workspace/ABE1"
OUT_DIR = "/workspace/abe1_out/full"
BACKBONE_DIR = "/workspace/hf/deberta-v3-large"


In [3]:
# The allocator flag has to be in the environment before torch first touches the GPU.
# 할당자 설정은 torch 가 GPU 를 처음 잡기 전에 환경에 있어야 한다.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import csv, datetime, gc, hashlib, inspect, json, math, platform, random, re, shutil, time
from pathlib import Path

# Turn off all warnings
# 경고 off
import warnings
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['PYTHONWARNINGS'] = 'ignore'

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
from sklearn.metrics import log_loss

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from safetensors.torch import load_file

import transformers
from transformers import logging as hf_logging
from transformers import (AutoTokenizer, DebertaV2Model, EarlyStoppingCallback,
                          Trainer, TrainerCallback, TrainingArguments)
from transformers.trainer_pt_utils import LengthGroupedSampler
from transformers.trainer_utils import get_last_checkpoint
hf_logging.set_verbosity_error()

print('python       :', platform.python_version())
print('torch        :', torch.__version__)
print('transformers :', transformers.__version__)
print('numpy/pandas :', np.__version__, '/', pd.__version__)

python       : 3.12.3
torch        : 2.6.0+cu124
transformers : 4.57.6
numpy/pandas : 2.2.6 / 2.3.3


### Environment check / GPU 점검

In [4]:
# GPU Configuration
# GPU 설정
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device        :', DEVICE)
print('cuda build    :', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
print('device count  :', torch.cuda.device_count())
assert torch.cuda.is_available(), 'No CUDA device — this notebook only trains on a GPU!!'

print('device name   :', torch.cuda.get_device_name(0))
print('capability    :', torch.cuda.get_device_capability(0))
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'VRAM          : {VRAM_GB:.1f} GiB')

# Cuda test
# Cuda 연산 테스트
print('GPU calculation is available', (torch.zeros(3, device='cuda') + 1).tolist())

# The backbone ships as a .bin, and transformers 4.57 refuses torch.load on torch < 2.6 (CVE-2025-32434)
# 백본이 .bin 이고, transformers 4.57 은 torch < 2.6 에서 .bin 의 torch.load 를 거부한다
assert tuple(int(v) for v in re.findall(r'\d+', torch.__version__)[:2]) >= (2, 6), \
    f'torch {torch.__version__} < 2.6 — install torch 2.6 or newer first!!'

if VRAM_GB < 40 and BATCH_SIZE >= 16:
    print(f'[warn] {VRAM_GB:.0f} GiB is less than the A40 this recipe was sized on — the worst-batch probe will '
          f'likely fail. Rerun with -p BATCH_SIZE 8 -p GRADIENT_ACCUMULATION_STEPS 2')

device        : cuda
cuda build    : 12.4
cuda available: True


device count  : 1
device name   : NVIDIA A40
capability    : (8, 6)
VRAM          : 44.4 GiB
GPU calculation is available [1.0, 1.0, 1.0]


### Seed / 시드

In [5]:
# It must be 42, since it is the ultimate answer to life, the universe and everything
SEED = 42
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f'Seed = {seed}')

set_seeds(SEED)

Seed = 42


### Directory setup / 경로

In [6]:
# Paths — everything the run writes goes under OUT_DIR, on the persistent volume
# 경로 설정 — 런이 쓰는 것은 전부 영구 볼륨 위 OUT_DIR 아래로 간다
PKG_DIR, OUT_DIR, BACKBONE_DIR = Path(PKG_DIR), Path(OUT_DIR), Path(BACKBONE_DIR)
DATA_DIR    = PKG_DIR / 'data'                 # frozen split csv + the recovered ids
RUN_DIR     = OUT_DIR / 'run'                  # Trainer checkpoints with optimizer state — for resuming
BEST_DIR    = OUT_DIR / 'best_checkpoints'     # BestSaver top-k, weights only
DELIVER_DIR = OUT_DIR / 'deliver'              # what comes home / 회수물
LOG_CSV     = OUT_DIR / 'train_log.csv'
DONE_FILE   = OUT_DIR / 'TRAINING_DONE.json'   # written when trainer.train() returns — a rerun skips to the export
OUT_DIR.mkdir(parents=True, exist_ok=True)

# A quick run and the full run must never share OUT_DIR — the full run would resume from the quick run's checkpoint.
# QUICK 런과 본 런이 OUT_DIR 을 같이 쓰면 본 런이 QUICK 체크포인트에서 '재개'해 버린다.
KIND = 'QUICK' if QUICK else 'FULL'
_kind_file = OUT_DIR / 'RUN_KIND'
if _kind_file.exists():
    assert _kind_file.read_text(encoding='utf-8').strip() == KIND, \
        f'{OUT_DIR} belongs to a {_kind_file.read_text(encoding="utf-8").strip()} run — point OUT_DIR somewhere else!!'
else:
    _kind_file.write_text(KIND, encoding='utf-8')

print(f'Mode          : {KIND}')
for _k in ('PKG_DIR', 'DATA_DIR', 'BACKBONE_DIR', 'OUT_DIR'):
    print(f'{_k:<14}= {globals()[_k]}')

Mode          : FULL
PKG_DIR       = /workspace/ABE1
DATA_DIR      = /workspace/ABE1/data
TOK_DIR       = /workspace/ABE1/tokenizer
BACKBONE_DIR  = /workspace/hf/deberta-v3-large
OUT_DIR       = /workspace/abe1_out/full


### Check assets / 자산 검사

In [7]:
# Asset check — every file present and byte-identical to the recorded hashes
# 자산 검사 — 파일이 다 있고, 기록된 해시와 바이트 단위로 같은가
# data/train.csv · valid.csv · holdout.csv are the very files ABE2 and TB2 trained (train) and selected (valid) with.
# data/train.csv · valid.csv · holdout.csv 는 ABE2·TB2 가 학습(train)하고 선택(valid)한 바로 그 파일이다.
SHA256 = {
    'data/train.csv':                  'cb1ae504f5d9a395008a10b33a53b8297c8e1aef158b098f1a07098ded791865',
    'data/valid.csv':                  '35b8291888ecbc3f3dbdd027a9f8654554db8e56259ab9a853320aeae2b14e9f',
    'data/holdout.csv':                '7145b5c8070c82cc63360df2dc56d9553ad5640b096eaad785e08092b70298ad',
    'data/recovered_ids.csv':          'fbe7bcf042bf4ba050ea60a1696b7fdb8a33bdb39310e2abffee2ee5a24f3638',
}
# microsoft/deberta-v3-large @ 64a8c8e (main). Its .bin and the hub's safetensors conversion hold identical tensors.
# spm.model · tokenizer_config.json are the official tokenizer files; cell 22 builds the tokenizer from them.
# microsoft/deberta-v3-large @ 64a8c8e(main). 이 .bin 과 허브의 safetensors 변환본은 텐서 402개가 전부 같다.
# spm.model · tokenizer_config.json 은 공식 토크나이저 파일이다 — 22번 셀이 이것으로 토크나이저를 만든다.
# The output below is the recorded run's, which also checked the tokenizer copy it loaded (9 package + 2 backbone files).
# 아래 출력은 기록된 런의 것이다 — 그 런은 읽어 들인 토크나이저 사본까지 검사했다(패키지 9 + 백본 2).
BACKBONE_SHA256 = {
    'config.json':           'ddec8b81d079d218ce9e54fc0af5d1d5937d6d53b5d42e70c1f251a1cebc830d',
    'pytorch_model.bin':     'dd5b5d93e2db101aaf281df0ea1216c07ad73620ff59c5b42dccac4bf2eef5b5',
    'spm.model':             'c679fbf93643d19aab7ee10c0b99e460bdbc02fedf34b92b05af343b4af586fd',
    'tokenizer_config.json': '3f3978e0c036f2c2588cac34a6047cbb0af0b0dc1814254e291028529805496d',
}

def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

# The backbone comes from the official repository at a pinned commit, fetched here when BACKBONE_DIR is empty.
# 백본은 공식 저장소의 고정 커밋에서 받는다 — BACKBONE_DIR 가 비어 있으면 여기서 받는다.
BACKBONE_REVISION = '64a8c8eab3e352a784c658aef62be1662607476f'
if not all((BACKBONE_DIR / k).exists() for k in BACKBONE_SHA256):
    from huggingface_hub import snapshot_download
    snapshot_download('microsoft/deberta-v3-large', revision=BACKBONE_REVISION, local_dir=BACKBONE_DIR,
                      allow_patterns=list(BACKBONE_SHA256))

_bad = []
for _p, _want in [(PKG_DIR / k, v) for k, v in SHA256.items()] + \
                 [(BACKBONE_DIR / k, v) for k, v in BACKBONE_SHA256.items()]:
    if not _p.exists():
        _bad.append(f'Missing {_p}')
    elif sha256(_p) != _want:
        _bad.append(f'sha256 mismatch: {_p}')

assert not _bad, 'Asset check failed!!\n  ' + '\n  '.join(_bad) + '\n  -> check PKG_DIR/data and BACKBONE_DIR'
print(f'*** Asset check passed *** ({len(SHA256)} package files + {len(BACKBONE_SHA256)} backbone files)')

*** Asset check passed *** (9 package files + 2 backbone files)


# 2. Load the split / 분할 불러오기

In [8]:
# Load the frozen split — read only, no train_test_split anywhere
# 동결 분할을 읽기만 한다 — train_test_split 은 어디에도 없다
train_df     = pd.read_csv(DATA_DIR / 'train.csv')          # 46,729 — the only rows that take a gradient
valid_df     = pd.read_csv(DATA_DIR / 'valid.csv')          # 5,000  — checkpoint selection only (2,000 clean + 3,000 recovered)
holdout_df   = pd.read_csv(DATA_DIR / 'holdout.csv')        # 5,748  — untouched until the report
recovered_df = pd.read_csv(DATA_DIR / 'recovered_ids.csv')  # 3,000 ids ABE1(V3) trained on — they go back to stage2

for d in (train_df, valid_df, holdout_df):
    # 정수 인코딩 / Integer label encoding: 0 = A wins, 1 = B wins, 2 = tie
    d['label'] = np.select(
        [d['winner_model_a'] == 1,
         d['winner_model_b'] == 1,
         d['winner_tie'] == 1],
        [0, 1, 2],
        default=-1,  # reserved for invalid or missing labels
    )

target_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
for name, d in (('train', train_df), ('valid', valid_df), ('holdout', holdout_df)):
    # 매핑 안 된 행(-1) 없나 / Ensure no rows fall back
    assert (d['label'] != -1).all(), f'{name}: invalid or missing labels!!'
    # 모든 행은 (1, 0, 0), (0, 1, 0), (0, 0, 1) 중 하나여야 한다 / every row is one-hot
    assert (d[target_cols].sum(axis=1) == 1).all(), f'{name}: rows that are not one-hot!!'
    print(f'{name:<8} {str(d.shape):<12} labels {d["label"].value_counts().sort_index().to_dict()}')
print('*** Label validation passed ***')

train    (46729, 10)  labels {0: 16312, 1: 15977, 2: 14440}
valid    (5000, 10)   labels {0: 1745, 1: 1710, 2: 1545}
holdout  (5748, 10)   labels {0: 2007, 1: 1965, 2: 1776}
*** Label validation passed ***


### Id sum check / Id 합 검산 — 3,000행 회수의 증명
분할별 행 수 · **id 합** · 라벨 수를 노트북에 박아 두고 읽은 파일과 대조한다. 박힌 값은 원본 csv 로 다시 계산해
대조한 값이다. 이 셀이 지키는 것은 셋이다.<br>
<br>
1. train 46,729행이 **다른 전문가들의 TR 과 같은 id** 다 (id 합 99,951,289,368,307)<br>
2. 회수 3,000행(id 합 6,480,182,614,921)이 **valid 안에만** 있고 train 에는 한 행도 없다<br>
3. 세 분할이 대회 train.csv 57,477행을 정확히 나눈다<br>
<br>
7절은 Trainer 에 실제로 넘어가는 행을 같은 값으로 **한 번 더** 검산한다.

In [9]:
# Id sum check — hardcoded. ABE1(V3) trained on 49,729 rows: TR 46,729 plus 3,000 rows of today's valid.
# Those 3,000 must sit in valid (selection only) and nowhere in train, or the recovery is void.
# Id 합 검산(하드코딩) — ABE1(V3)는 49,729행(TR 46,729 + 지금 valid 의 3,000행)으로 학습했다.
# 그 3,000행이 valid(선택 전용)에만 있고 train 에는 한 행도 없어야 회수가 성립한다.
EXPECT = {               #   rows,             id sum,   labels (A, B, tie)
    'train':     (46_729, 99_951_289_368_307, (16_312, 15_977, 14_440)),   # TR — ABE2·TB1·TB2·Arith trained on these ids
    'valid':     ( 5_000, 10_712_838_655_783, ( 1_745,  1_710,  1_545)),
    'holdout':   ( 5_748, 12_483_999_372_887, ( 2_007,  1_965,  1_776)),
    'recovered': ( 3_000,  6_480_182_614_921, ( 1_047,  1_026,    927)),   # ABE1(V3) trained on these — recovered now
    'clean':     ( 2_000,  4_232_656_040_862, (   698,    684,    618)),   # ABE1(V3)'s own valid
}
ALL_ROWS, ALL_SUM = 57_477, 123_148_127_396_977                            # the competition train.csv

_rec = set(recovered_df['id'])
_sets = {'train': set(train_df['id']), 'valid': set(valid_df['id']), 'holdout': set(holdout_df['id'])}
_frames = {'train': train_df, 'valid': valid_df, 'holdout': holdout_df,
           'recovered': valid_df[valid_df['id'].isin(_rec)],
           'clean': valid_df[~valid_df['id'].isin(_rec)]}

print(f'{"item":<10}{"rows":>8}{"id sum":>22}   labels (A, B, tie)')
for _k, _d in _frames.items():
    _got = (len(_d), int(_d['id'].sum()), tuple(int((_d['label'] == c).sum()) for c in (0, 1, 2)))
    print(f'{_k:<10}{_got[0]:>8,}{_got[1]:>22,}   {_got[2]}')
    assert _got == EXPECT[_k], f'{_k}: got {_got}, expected {EXPECT[_k]}!!'

for _k, _s in _sets.items():
    assert len(_s) == len(_frames[_k]), f'{_k}: duplicated ids!!'
assert len(_rec) == len(recovered_df) == EXPECT['recovered'][0], 'recovered_ids.csv is not 3,000 unique ids!!'
assert _rec <= _sets['valid'], f'{len(_rec - _sets["valid"])} recovered ids are not in valid!!'
assert not (_rec & _sets['train']), f'{len(_rec & _sets["train"])} recovered ids are in train — ABE1 would learn them again!!'
assert not (_sets['train'] & _sets['valid']), 'train and valid overlap!!'
assert not (_sets['train'] & _sets['holdout']), 'train and holdout overlap!!'
assert not (_sets['valid'] & _sets['holdout']), 'valid and holdout overlap!!'
assert sum(EXPECT[k][0] for k in _sets) == ALL_ROWS and sum(EXPECT[k][1] for k in _sets) == ALL_SUM, \
    'the three splits do not add up to the competition train.csv!!'
assert EXPECT['clean'][1] + EXPECT['recovered'][1] == EXPECT['valid'][1], 'valid = clean + recovered does not add up!!'

print('\n*** Id check passed ***')
print('    train 46,729 = the TR of ABE2 · TB1 · TB2 · Arith, id for id')
print('    the 3,000 recovered rows sit in valid only — selection, no gradient — and rejoin stage2 afterwards')
print(f'    stage2 grows 7,748 -> {EXPECT["valid"][0] + EXPECT["holdout"][0]:,} rows (valid 5,000 + holdout 5,748)')

item          rows                id sum   labels (A, B, tie)
train       46,729    99,951,289,368,307   (16312, 15977, 14440)
valid        5,000    10,712,838,655,783   (1745, 1710, 1545)
holdout      5,748    12,483,999,372,887   (2007, 1965, 1776)
recovered    3,000     6,480,182,614,921   (1047, 1026, 927)
clean        2,000     4,232,656,040,862   (698, 684, 618)

*** Id check passed ***
    train 46,729 = the TR of ABE2 · TB1 · TB2 · Arith, id for id
    the 3,000 recovered rows sit in valid only — selection, no gradient — and rejoin stage2 afterwards
    stage2 grows 7,748 -> 10,748 rows (valid 5,000 + holdout 5,748)


### QUICK slice / 배선 확인용 슬라이스

In [10]:
# QUICK slices the frames AFTER the id check, so the check always sees the full splits.
# The 32 longest training rows ride along, so the worst-batch probes test the real 2048-token batches.
# QUICK 은 id 검산 뒤에 자른다 — 검산은 언제나 전체 분할을 본다.
# 가장 긴 학습 행 32개를 함께 넣어 최악배치 프로브가 실제 2048 토큰 배치를 재게 한다.
if QUICK:
    # branch length proxy in characters: prompt + the longer response / 브랜치 길이의 글자 수 대리값
    _len = train_df['prompt'].str.len() + \
        train_df[['response_a', 'response_b']].apply(lambda c: c.str.len()).max(axis=1)
    _long = _len.nlargest(32).index
    _rest = train_df.drop(index=_long).sample(480, random_state=SEED).index
    train_df   = train_df.loc[_long.append(_rest)].reset_index(drop=True)
    valid_df   = valid_df.sample(256, random_state=SEED).reset_index(drop=True)
    holdout_df = holdout_df.sample(256, random_state=SEED).reset_index(drop=True)
    print(f'QUICK slice — train {len(train_df)} (32 longest + 480) / valid {len(valid_df)} / holdout {len(holdout_df)}')
else:
    print(f'FULL — train {len(train_df):,} / valid {len(valid_df):,} / holdout {len(holdout_df):,}')

FULL — train 46,729 / valid 5,000 / holdout 5,748


# 3. Tokenizing / 토크나이징

In [11]:
# The backbone is microsoft/deberta-v3-large, read from BACKBONE_DIR at the pinned commit (cell 14)
# 백본은 microsoft/deberta-v3-large — 14번 셀이 고정 커밋으로 받아 둔 BACKBONE_DIR 에서 읽는다
MODEL_NAME = 'microsoft/deberta-v3-large'

MAX_LEN   = 2048
Q_FLOOR   = 256

# 잘릴 때 앞 60% + 뒤 40%
# if sequences are truncated, keep 60% from the front and 40% from the rear to preserve context
HEAD_FRAC = 0.6

# The official tokenizer of the backbone snapshot (spm.model · tokenizer_config.json) plus the [BR] token.
# The recorded run below loaded the copy V3 saved after adding [BR]; the two give the same ids on 30,000 + 9,000 texts.
# 백본 스냅샷의 공식 토크나이저(spm.model · tokenizer_config.json)에 [BR] 를 더한다.
# 아래 출력을 낸 런은 V3 가 [BR] 를 더해 저장한 사본을 읽었고, 두 토크나이저는 텍스트 30,000 + 9,000개에서 id 가 전부 같다.
tokenizer = AutoTokenizer.from_pretrained(BACKBONE_DIR)
tokenizer.add_tokens(['[BR]'])
CLS, SEP, PAD = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id

# 입력 구조 :  [CLS] Q [SEP] ANSWER_A [SEP] / [CLS] Q [SEP] ANSWER_B [SEP]
# input sequences : [CLS] Q [SEP] ANSWER_A [SEP] / [CLS] Q [SEP] ANSWER_B [SEP]
BUDGET = MAX_LEN - 3
assert (CLS, SEP, PAD) == (1, 2, 0), f'CLS/SEP/PAD {(CLS, SEP, PAD)} != (1, 2, 0)!!'
print(f'{type(tokenizer).__name__} · vocab {len(tokenizer):,} · CLS/SEP/PAD {CLS}/{SEP}/{PAD} · budget {BUDGET}')

DebertaV2TokenizerFast · vocab 128,002 · CLS/SEP/PAD 1/2/0 · budget 2045


### 개행 문자 보존 / Keep the line breaks
토크나이저가 지우는 개행을 `[BR]` 로 바꿔 보존한다 — 개행 빈도가 선택에 영향을 준다는 V3 의 관찰.

In [12]:
# replace \n(line break) to [BR] so that the tokenizer does not erase it (since it is the primary feature)
# so that the model can learn [BR]'s effect directly.
# V3 called add_tokens here. The tokenizer above already carries [BR], so the same call now adds nothing.
# V3 는 여기서 add_tokens 를 불렀다. 위 셀의 토크나이저에 [BR] 가 이미 있으므로 같은 호출이 아무것도 더하지 않는다.
_n_new = tokenizer.add_tokens(['[BR]'])
BR_ID = tokenizer.convert_tokens_to_ids('[BR]')
assert (_n_new, BR_ID, len(tokenizer)) == (0, 128001, 128002), \
    f'[BR] drifted: added {_n_new}, id {BR_ID}, vocab {len(tokenizer)} (V3: 0 / 128001 / 128002)!!'

# verify
# 토크나이저가 개행을 어떻게 처리하는지 확인
_raw = 'first line\nsecond line\n\nnew paragraph'

def to_model_text(t):
    return t.replace('\n', ' [BR] ')

print('Before :', tokenizer.convert_ids_to_tokens(
    tokenizer(_raw, add_special_tokens=False)['input_ids']))
print('After preservation :', tokenizer.convert_ids_to_tokens(
    tokenizer(to_model_text(_raw), add_special_tokens=False)['input_ids']))

Before : ['▁first', '▁line', '▁second', '▁line', '▁new', '▁paragraph']
After preservation : ['▁first', '▁line', '▁', '[BR]', '▁second', '▁line', '▁', '[BR]', '▁', '[BR]', '▁new', '▁paragraph']


### Truncation / 절단

In [13]:
# V3 verbatim — the same truncation the deployed ABE1 was trained with
# V3 그대로 — 배포 중인 ABE1 이 학습한 것과 같은 절단
def parse_text(x):
    if pd.isna(x):
        return ''
    try:
        turns = json.loads(x)
    except (json.JSONDecodeError, TypeError):
        return str(x)
    if isinstance(turns, list):
        return ' '.join(t for t in turns if isinstance(t, str))
    return str(turns)

# If truncated, preserve k tokens for Question then extract 60% from front, 40% from rear for every Answer
# k개만 남기되 앞 60% + 뒤 40% 추출
def head_tail(ids, k):
    ids = list(ids)                 # implicit casting for unattended dataframe input
    if len(ids) <= k:
        return ids
    h = int(k * HEAD_FRAC)
    t = k - h
    return ids[:h] + ids[-t:] if t > 0 else ids[:h]

# 답변이 안 쓴 예산은 Q가 회수.
# allocate leftover budget for Question
def alloc_q(lq, l_resp):
    kq = min(lq, Q_FLOOR)
    k_resp = min(l_resp, BUDGET - kq)
    slack = BUDGET - kq - k_resp
    return min(lq, kq + slack)

# 길이가 다른 입력에서 동일한 질문 입력 보장
# ensures both branches keep the identical question prefix even when answers of different lengths are truncated
def build_row(iq, ia, ib):
    lq, la, lb = len(iq), len(ia), len(ib)
    kq = alloc_q(lq, max(la, lb))
    ka = min(la, BUDGET - kq)
    kb = min(lb, BUDGET - kq)
    q = head_tail(iq, kq)
    sa = [CLS] + q + [SEP] + head_tail(ia, ka) + [SEP]
    sb = [CLS] + q + [SEP] + head_tail(ib, kb) + [SEP]
    return sa, sb

### Dataset / 데이터셋 정의

In [14]:
# 전처리 함수, 학습 전에 전부 돌려놔서 메모리에 상주시키면 학습 시 CPU 오버헤드가 일어나지 않는다
# Tokenize everything up front and keep it in memory to avoid CPU overhead during training (~150 MB of RAM)
# V3 verbatim, plus `ids` so the rows handed to the Trainer can be audited by id sum
# V3 그대로 + `ids` — Trainer 에 넘어가는 행을 id 합으로 감사하기 위해
class PrefDataset(Dataset):
    def __init__(self, df, has_label=True, swap_aug=False):
        df = df.reset_index(drop=True)
        self.has_label = has_label
        self.swap_aug = swap_aug          # 학습셋만 True
        self.ids = df['id'].to_numpy()

        q_raw = df['prompt'].map(parse_text).tolist()
        a_raw = df['response_a'].map(parse_text).tolist()
        b_raw = df['response_b'].map(parse_text).tolist()

        iq = tokenizer([to_model_text(t) for t in q_raw], add_special_tokens=False)['input_ids']
        ia = tokenizer([to_model_text(t) for t in a_raw], add_special_tokens=False)['input_ids']
        ib = tokenizer([to_model_text(t) for t in b_raw], add_special_tokens=False)['input_ids']

        self.seq_a, self.seq_b = [], []
        for x, y, z in zip(iq, ia, ib):
            sa, sb = build_row(x, y, z)
            self.seq_a.append(np.asarray(sa, dtype=np.int32))
            self.seq_b.append(np.asarray(sb, dtype=np.int32))

        # for length sorted batching
        # 길이 정렬 배칭 키 = max(La, Lb) — 스왑 불변
        self.lengths = [max(len(x), len(y)) for x, y in zip(self.seq_a, self.seq_b)]
        self.labels = df['label'].tolist() if has_label else None

    def __len__(self):
        return len(self.seq_a)

    def __getitem__(self, i):
        a, b = self.seq_a[i], self.seq_b[i]
        lab = self.labels[i] if self.has_label else None
        # Swap augment - There is a 50% chance of swap the labels of the A and B
        # it is harmless because this dataset barely has position bias(34.91% vs 34.19%)
        # 위치 편향이 거의 없는 데이터셋이므로, 50% 확률로 A, B의 위치를 바꿔서 데이터를 보정한다.
        if self.swap_aug and random.random() < 0.5:
            a, b = b, a
            if lab is not None and lab != 2:     # tie(2)는 불변
                lab = 1 - lab
        item = {'a_ids': a, 'b_ids': b}
        if lab is not None:
            item['labels'] = lab
        return item


# 동적 패딩, 0으로 패딩된 부분은 학습 시 어텐션 연산의 대상이 되지 않음.
# dynamic padding, padding parts are not multiplied during learning process.
def collate_fn(batch):
    out = {}
    for key, mkey in [('a_ids', 'a_mask'), ('b_ids', 'b_mask')]:
        seqs = [b[key] for b in batch]
        L = max(len(s) for s in seqs)
        ids  = np.full((len(seqs), L), PAD, dtype=np.int64)
        mask = np.zeros((len(seqs), L), dtype=np.int64)
        for i, s in enumerate(seqs):
            ids[i, :len(s)] = s
            mask[i, :len(s)] = 1
        out[key]  = torch.from_numpy(ids)
        out[mkey] = torch.from_numpy(mask)
    if 'labels' in batch[0]:
        out['labels'] = torch.tensor([b['labels'] for b in batch], dtype=torch.long)
    return out

# 4. Model / 모델 정의

### Model — V3 as it was / 모델 — V3 그대로
헤드는 `LayerNorm(3072) → Dropout → Linear(3072, 1024) → GELU → Dropout → Linear(1024, 3)`, 입력은 `[a, b, a−b]` 이다.
4블록 헤드(ABE2 형)·풀링 초기화 변경·LR 5e-6 은 V4 런에서 한 묶음으로 돌아 같은 holdout 에서 **+0.0097 졌다**.
동결 임베딩 위 헤드 스윕의 승자들은 end-to-end 로 확인된 적이 없다.<br>
<br>
키 이름이 최종 노트북의 `build_model('abe1')` 과 정확히 같아서, 회수한 가중치는 그쪽 코드 수정 없이 로드된다.

In [15]:
# V3 verbatim — zero-init attention pooling and the 3-block head [a, b, a-b].
# The state_dict keys are exactly what the final notebook's build_model('abe1') loads — no code there changes.
# V3 그대로 — 0 초기화 어텐션 풀링과 3블록 헤드 [a, b, a-b].
# state_dict 키가 최종 노트북의 build_model('abe1') 이 읽는 것과 정확히 같다 — 그쪽 코드는 바뀌지 않는다.

# Attention pooling — scorer zero-init, only increase if bias could improve output
class AttnPool(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.score = nn.Linear(h, 1)
        nn.init.zeros_(self.score.weight)
        nn.init.zeros_(self.score.bias)
        self.force_uniform = False        # True → mean pooling

    def forward(self, last_hidden, mask):
        s = self.score(last_hidden).squeeze(-1)
        if self.force_uniform:
            s = torch.zeros_like(s)
        s = s.float().masked_fill(mask == 0, float('-inf'))
        w = torch.softmax(s, dim=1).unsqueeze(-1)
        return (last_hidden.float() * w).sum(dim=1)

# 모델 생성자 — 순수 텍스트 모델 (산술 피처는 사후 스태킹 단계로 이동)
# model ctr
class PrefModel(nn.Module):
    def __init__(self, model_name, num_classes=3,
                 vocab_size=None, new_token_ids=()):
        super().__init__()
        self.encoder, info = DebertaV2Model.from_pretrained(
            model_name, dtype=torch.float32, output_loading_info=True)      # implicit fp32 casting
        # Warnings are off — without this, a wrong path would train a random-init encoder for 9 hours.
        # 경고를 꺼 두었으므로, 이 검사가 없으면 경로가 틀려도 무작위 초기화 인코더로 9시간을 학습한다.
        assert not info['missing_keys'], f"encoder weights missing: {info['missing_keys'][:5]}!!"
        print(f"encoder loaded — {len(info['unexpected_keys'])} unused key(s) dropped "
              f"(pretraining heads: {sorted({k.split('.')[0] for k in info['unexpected_keys']})})")

        # [BR] 토큰을 위한 임베딩 초기화
        # initialize the [BR] embedding
        if new_token_ids:
            emb = self.encoder.get_input_embeddings()
            if vocab_size is not None and vocab_size > emb.weight.size(0):
                self.encoder.resize_token_embeddings(vocab_size)
                emb = self.encoder.get_input_embeddings()
            with torch.no_grad():
                mean_vec = emb.weight[:min(new_token_ids)].mean(0)  # 실제 사전학습된 행들의 평균
                for tid in new_token_ids:
                    emb.weight[tid] = mean_vec

        h = self.encoder.config.hidden_size       # large = 1024
        self.pool = AttnPool(h)                   # 학습되는 풀링 — 체크포인트에 scorer 포함됨
        in_dim = h * 3                            # [a, b, a-b] = 3072
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(0.1),
            nn.Linear(in_dim, 1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, num_classes),
        )
        self.loss_fct = nn.CrossEntropyLoss()

    def gradient_checkpointing_enable(self, **kwargs):
        self.encoder.gradient_checkpointing_enable(**kwargs)

    def _encode(self, ids, mask):
        out = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state
        return self.pool(out, mask)

    def forward(self, a_ids, a_mask, b_ids, b_mask, labels=None):
        # autocast는 스레드 로컬 → DataParallel 전파를 위해 forward 내부에서
        with torch.autocast(device_type='cuda', dtype=torch.float16,
                            enabled=torch.cuda.is_available()):
            a = self._encode(a_ids, a_mask)
            b = self._encode(b_ids, b_mask)
            x = torch.cat([a, b, a - b], dim=1)
            logits = self.head(x)

        logits = logits.float()                       # 손실/지표는 fp32 로
        if labels is None:
            return {'logits': logits}                 # loss 키 자체를 넣지 않음
        return {'loss': self.loss_fct(logits, labels), 'logits': logits}

# 5. Training setup / 학습 설정

In [16]:
# remove any unused variables, basically remove trainer, model class and extra
# 기본으로 trainer/model 을 지우고, extra 로 넘긴 변수명도 함께 삭제
def mem_free(*extra):
    for g in ['trainer', 'model', *extra]:
        if g in globals():
            del globals()[g]
    # PyTorch 모델은 순환참조가 있어 여러 번 돌려 확실히 회수
    # to break circular references
    for _ in range(3):
        gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print('memory cleared')

mem_free()

memory cleared


### Hyperparameters / 하이퍼파라미터

In [17]:
# Hyperparameter setup — the V3 recipe. What differs from V3 is only when it evaluates and what it keeps.
# 하이퍼파라미터 — V3 레시피 그대로. V3 와 다른 것은 언제 평가하고 무엇을 남기는가뿐이다.
LR           = 1e-5      # V3. The 5e-6 recipe ran in V4 and lost on the holdout
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EPOCHS       = 3         # V3's schedule length — EarlyStopping usually ends the run near epoch 2
assert BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS == 16, 'the effective batch must stay 16 (V3)!!'

STEPS_PER_EPOCH = math.ceil(len(train_df) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))   # 2,921 in the full run
# Quarter-epoch evals. The 4th lands on step 2,920 — one step short of V3's pick, the end of epoch 1.
# ¼ 에폭마다 평가 — 4번째가 step 2,920 으로, V3 가 고른 '1 에폭 끝'과 한 스텝 차이다.
EVAL_STEPS  = 730
ES_PATIENCE = 5          # 5 x 730 steps ≈ 1.25 epochs without a new best -> stop (the V4/V5 rule)
KEEP_TOP_K  = 3          # BestSaver: the best three by valid, weights only
MAX_STEPS   = -1
if QUICK:                # the same code paths on a slice: three evals, one BestSaver eviction, an ES that can fire
    EVAL_STEPS, ES_PATIENCE, KEEP_TOP_K, MAX_STEPS = 8, 2, 2, 24

USE_COMPILE = False

training_args = TrainingArguments(
    output_dir=str(RUN_DIR),
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,      # 평가는 no-grad라 학습 배치보다 크게
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=EVAL_STEPS,
    save_total_limit=1,                              # + the best one, which load_best_model_at_end protects
    load_best_model_at_end=True,
    metric_for_best_model='eval_logloss',
    greater_is_better=False,

    fp16=True,                                       # V3: fp16 + GradScaler
    bf16=False,
    gradient_checkpointing=True,                     # the 2048-token worst batch needs it even on 48GB
    gradient_checkpointing_kwargs={'use_reentrant': False},
    # keep optimizer/scheduler/RNG state so an interrupted run can resume (~5GB per checkpoint)
    # 중단 대비: 옵티마이저·스케줄러·RNG 상태까지 저장해야 resume 가 된다
    save_only_model=False,

    dataloader_num_workers=4,
    eval_accumulation_steps=10,
    logging_steps=4 if QUICK else 50,
    report_to='none',
    disable_tqdm=False,
    remove_unused_columns=False,
    torch_compile=USE_COMPILE,
    seed=SEED,
)

_total = MAX_STEPS if MAX_STEPS > 0 else STEPS_PER_EPOCH * EPOCHS
print(f"""
==================================================
[Dataset]
- Training set  : {len(train_df):,} rows   (gradient)
- Valid set     : {len(valid_df):,} rows   (selection only — 2,000 clean + 3,000 recovered)
- Hold-out set  : {len(holdout_df):,} rows   (report only)

[Optimization — the V3 recipe]
- schedule                    : {EPOCHS} epochs · {STEPS_PER_EPOCH:,} steps/epoch · {_total:,} steps
- BATCH_SIZE x GA             : {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = 16
- LEARNING_RATE               : {LR}
- WEIGHT_DECAY                : {WEIGHT_DECAY}
- WARMUP_RATIO                : {WARMUP_RATIO}
- MAX_GRAD_NORM               : {training_args.max_grad_norm}
- MAX_LEN                     : {MAX_LEN}
- SEED                        : {SEED}

[Evaluation & selection — fixed before training]
- eval / save every           : {EVAL_STEPS} steps (≈ {EVAL_STEPS / STEPS_PER_EPOCH:.2f} epoch)
- pick                        : the lowest eval_logloss on valid {len(valid_df):,} (forward view)
- EarlyStopping patience      : {ES_PATIENCE} evals
- BestSaver top-k             : {KEEP_TOP_K}

[System & precision]
- fp16 / bf16                 : {training_args.fp16} / {training_args.bf16}   (GradScaler on)
- gradient checkpointing      : {training_args.gradient_checkpointing}
- eval batch                  : {EVAL_BATCH_SIZE}
- output directory            : {OUT_DIR}
==================================================
""")


[Dataset]
- Training set  : 46,729 rows   (gradient)
- Valid set     : 5,000 rows   (selection only — 2,000 clean + 3,000 recovered)
- Hold-out set  : 5,748 rows   (report only)

[Optimization — the V3 recipe]
- schedule                    : 3 epochs · 2,921 steps/epoch · 8,763 steps
- BATCH_SIZE x GA             : 16 x 1 = 16
- LEARNING_RATE               : 1e-05
- WEIGHT_DECAY                : 0.01
- WARMUP_RATIO                : 0.1
- MAX_GRAD_NORM               : 1.0
- MAX_LEN                     : 2048
- SEED                        : 42

[Evaluation & selection — fixed before training]
- eval / save every           : 730 steps (≈ 0.25 epoch)
- pick                        : the lowest eval_logloss on valid 5,000 (forward view)
- EarlyStopping patience      : 5 evals
- BestSaver top-k             : 3

[System & precision]
- fp16 / bf16                 : True / False   (GradScaler on)
- gradient checkpointing      : True
- eval batch                  : 32
- output directory         

### Logger & metrics / 로거와 지표

In [18]:
# Custom CSV logger called in every evaluation, basically supporting multiple gpus with is_process_zero
# 채점 시마다 호출되어 학습률, 평가 결과 등을 저장하는 CSV 로거 콜백, 다중 GPU의 경우 첫 번째 GPU에서만 기록함.
class TrainingLogger(TrainerCallback):
    def __init__(self, filepath):
        self.filepath = filepath
        self.fieldnames = [
            'step', 'epoch', 'loss', 'learning_rate',
            'eval_loss', 'eval_logloss', 'eval_accuracy',
        ]
        self._seen = set()   # for dataparallel

    def on_train_begin(self, args, state, control, **kwargs):
        if state.is_world_process_zero:
            self._seen.clear()
            os.makedirs(os.path.dirname(self.filepath), exist_ok=True)
            # 파일이 비어있으면 헤더 작성
            if not os.path.exists(self.filepath) or os.path.getsize(self.filepath) == 0:
                with open(self.filepath, 'a', newline='', encoding='utf-8') as f:
                    csv.DictWriter(f, fieldnames=self.fieldnames, extrasaction='ignore').writeheader()

    # logging protocol
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not (state.is_world_process_zero and logs is not None):
            return
        # 로그 종류 판별: eval 로그 / train 로그 / 그 외(skip)
        if 'eval_loss' in logs:
            kind = 'eval'
        elif 'loss' in logs:
            kind = 'train'
        else:
            return
        # 같은 step의 같은 종류 로그가 또 오면(Trainer 중복 방출) 건너뜀
        key = (state.global_step, kind)
        if key in self._seen:
            return
        self._seen.add(key)

        row = {k: '' for k in self.fieldnames}
        row['step'] = state.global_step
        row['epoch'] = logs.get('epoch', state.epoch)
        for k in self.fieldnames:
            if k in logs:
                row[k] = logs[k]
        with open(self.filepath, 'a', newline='', encoding='utf-8') as f:
            csv.DictWriter(f, fieldnames=self.fieldnames, extrasaction='ignore').writerow(row)


# we will record log_loss and accuracy for this logger
# 지표 : log_loss + accuracy
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = torch.tensor(np.array(logits), dtype=torch.float32)
    probs = torch.softmax(logits, dim=1).numpy()
    preds = probs.argmax(axis=1)
    return {
        'logloss':  log_loss(labels, probs, labels=[0, 1, 2]),
        'accuracy': (preds == labels).mean(),
    }

### BestSaver — keep the top-k by valid / 상위 k 보존

In [19]:
# BestSaver — the ABE2/TB2 callback, verbatim
# Trainer's save_total_limit keeps only the MOST RECENT checkpoints (sorted by step, not by score) — the best one is
# protected but mid-run good ones get deleted. This callback copies the top-K scorers into a separate folder.
# Weights only: optimizer/scheduler/RNG are skipped.
# Trainer의 save_total_limit은 '스텝 순 최신' 기준이라 중간 구간의 좋은 체크포인트가 지워진다.
# 이 콜백은 점수 상위 K개를 별도 폴더에 복사한다. 가중치만 남긴다.
class BestSaver(TrainerCallback):
    def __init__(self, k=5, output_dir=None, metric='eval_logloss', greater_is_better=False):
        self.k = k
        self.metric = metric                          # must match metric_for_best_model
        self.sign = -1 if greater_is_better else 1    # sort ascending on sign * score
        self.skip = shutil.ignore_patterns('optimizer.pt', 'scheduler.pt',
                                           'rng_state*.pth', 'scaler.pt')
        self.best_dir = os.path.join(output_dir, 'best_checkpoints')
        self.state_file = os.path.join(self.best_dir, 'best_k_state.json')
        self.best_checkpoints = []
        os.makedirs(self.best_dir, exist_ok=True)
        self._load_state()

    def _load_state(self):
        # Restore the ranking so a resumed run does not start counting over
        # 재개 시 랭킹 복구 — 처음부터 다시 세지 않도록
        if os.path.exists(self.state_file):
            try:
                with open(self.state_file, 'r', encoding='utf-8') as f:
                    self.best_checkpoints = json.load(f)
                print(f'[BestSaver] state restored ({len(self.best_checkpoints)} entries)')
            except Exception as e:
                print(f'[BestSaver] state load failed, starting fresh: {e}')

    def _save_state(self):
        for rank, entry in enumerate(self.best_checkpoints, 1):
            entry['rank'] = rank
        try:
            with open(self.state_file, 'w', encoding='utf-8') as f:
                json.dump(self.best_checkpoints, f, indent=4, ensure_ascii=False)
        except Exception as e:
            print(f'[BestSaver] state save failed: {e}')

    def on_save(self, args, state, control, **kwargs):
        if not state.is_world_process_zero:
            return

        src = os.path.join(args.output_dir, f'checkpoint-{state.global_step}')
        if not os.path.isdir(src):
            raise FileNotFoundError(f'[BestSaver] checkpoint not found: {src}')

        # Look up this step's score. With eval_steps == save_steps the Trainer evaluates
        # before it saves, so the entry is already in log_history.
        # 이번 스텝의 점수 조회 — eval_steps == save_steps면 평가가 저장보다 먼저 실행된다.
        score = None
        for log in reversed(state.log_history):
            if log.get('step') == state.global_step and self.metric in log:
                score = log[self.metric]
                break
        if score is None:                                    # fallback / 폴백
            for log in reversed(state.log_history):
                if log.get('step') == state.global_step and 'eval_loss' in log:
                    score = log['eval_loss']
                    break
        if score is None:
            # Warn rather than raise: losing a 9-hour run to a logging hiccup is not worth it
            # 예외 대신 경고 — 로깅 타이밍 문제로 9시간짜리 런을 잃을 이유는 없다
            print(f'[BestSaver] no eval score at step {state.global_step}, skipped')
            return

        entry = {
            'score': score,
            'metric': self.metric,
            'step': state.global_step,
            'epoch': round(state.epoch, 4) if state.epoch else 0,
            'saved_at': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        }
        self.best_checkpoints.append(entry)
        self.best_checkpoints.sort(key=lambda x: self.sign * x['score'])
        self._save_state()

        # Copy in if this step made the top-K / top-K에 들면 복사
        if any(e['step'] == state.global_step for e in self.best_checkpoints[:self.k]):
            dest = os.path.join(self.best_dir, f'checkpoint-{state.global_step}')
            if not os.path.exists(dest):
                rank = self.best_checkpoints.index(entry) + 1
                print(f'[BestSaver] rank {rank} ({self.metric}={score:.4f}) -> {dest}')
                for i in range(3):          # retry — the copy can race with Trainer's rotation
                    try:
                        shutil.copytree(src, dest, ignore=self.skip)
                        break
                    except Exception as e:
                        shutil.rmtree(dest, ignore_errors=True)
                        if i == 2:
                            print(f'[BestSaver] copy failed after retries: {e}')
                            raise
                        time.sleep(3)

        # Delete whatever was pushed out of the top-K / 상위 K에서 밀려난 사본 삭제
        if len(self.best_checkpoints) > self.k:
            dropped = self.best_checkpoints[self.k:]
            self.best_checkpoints = self.best_checkpoints[:self.k]
            self._save_state()
            for e in dropped:
                path = os.path.join(self.best_dir, f'checkpoint-{e["step"]}')
                if os.path.exists(path):
                    print(f'[BestSaver] removing {e["score"]:.4f} @ step {e["step"]}')
                    shutil.rmtree(path, ignore_errors=True)

### Length-sorted samplers / 길이 정렬 샘플러

In [20]:
# Custom samplers — length-grouped batches for training (V3), length-sorted batches for eval (ABE2/TB2)
# 커스텀 샘플러 — 학습은 길이 그룹 배칭(V3), 평가는 길이 정렬 배칭(ABE2/TB2) — 패딩 낭비 제거
class GroupedTrainer(Trainer):
    def _get_train_sampler(self, train_dataset=None):
        ds = train_dataset if train_dataset is not None else self.train_dataset
        return LengthGroupedSampler(
            batch_size=self.args.train_batch_size * self.args.gradient_accumulation_steps,
            dataset=ds, lengths=ds.lengths, model_input_name=None,
        )

    def _get_eval_sampler(self, eval_dataset):
        return np.argsort(np.asarray(eval_dataset.lengths)).tolist()

# if the parent signature differs, the override is silently ignored and eval stays slow
# 부모 시그니처가 다르면 override 가 조용히 무시된다.
for name in ['_get_train_sampler', '_get_eval_sampler']:
    base, mine = inspect.signature(getattr(Trainer, name)), inspect.signature(getattr(GroupedTrainer, name))
    assert list(base.parameters) == list(mine.parameters), f'{name} signature mismatch — sampler override not applied!!'
print('*** Length-sorted samplers active ***')

*** Length-sorted samplers active ***


### Datasets / 데이터셋 생성

In [21]:
# Enable swap augment only on the training set
# train set에서만 Swap 증강 활성화
_t0 = time.time()
train_ds   = PrefDataset(train_df, swap_aug=True)
valid_ds   = PrefDataset(valid_df)
holdout_ds = PrefDataset(holdout_df)
print(f'tokenized {len(train_ds) + len(valid_ds) + len(holdout_ds):,} rows in {time.time() - _t0:.0f}s')

for _n, _ds in (('train', train_ds), ('valid', valid_ds), ('holdout', holdout_ds)):
    _L = np.asarray(_ds.lengths)
    print(f'  {_n:<8} rows {len(_ds):>6,} · max(La, Lb) p50 {int(np.median(_L)):>4} · p99 {int(np.percentile(_L, 99)):>4} · '
          f'at {MAX_LEN}: {(_L >= MAX_LEN).mean() * 100:.1f}%')

tokenized 57,477 rows in 30s
  train    rows 46,729 · max(La, Lb) p50  378 · p99 2048 · at 2048: 1.9%
  valid    rows  5,000 · max(La, Lb) p50  377 · p99 2048 · at 2048: 1.9%
  holdout  rows  5,748 · max(La, Lb) p50  381 · p99 2048 · at 2048: 1.8%


# 6. Smoke tests / 스모크 테스트

### Smoke tests / 스모크 테스트
① 전처리 불변식 3,000행 ② 라벨 없는 predict ③ 학습 최악배치(16 × 2048) 1스텝 ③b 평가 최악배치(32 × 2048).
③·③b 는 실제 Trainer 경로로 잰다. 프로브가 바꾼 가중치는 스냅샷으로 되돌리고, 프로브 체크포인트는 남기지 않는다.

In [22]:
# 0) Construct model, trainer / 모델, 트레이너 객체 생성
model = PrefModel(str(BACKBONE_DIR), vocab_size=len(tokenizer), new_token_ids=[BR_ID]).to(DEVICE)
trainer = GroupedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=valid_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

# 1) Preprocessing invariant over 3,000 rows: both branches share the identical question prefix, and neither exceeds MAX_LEN
# 1) 전처리 불변식 3,000행 전수: 두 브랜치의 Q 프리픽스 동일 + 길이 한도 준수
for sa, sb in zip(train_ds.seq_a[:3000], train_ds.seq_b[:3000]):
    qa = list(sa[:list(sa).index(SEP)])
    qb = list(sb[:list(sb).index(SEP)])
    assert qa == qb, 'Asymmetric question prefix between the a/b branches!!'
    assert len(sa) <= MAX_LEN and len(sb) <= MAX_LEN, 'Sequence longer than MAX_LEN!!'

# 2) Label-free predict through the Trainer — catches head/collator wiring bugs before the long run
# 2) Trainer 경유 라벨 없는 predict — 헤드·콜레이터 배선 버그를 잡는다
_smoke = PrefDataset(train_df.iloc[:4], has_label=False)
_ = trainer.predict(_smoke)
print('Smoke tests 1 & 2 passed')

# 3) Worst-batch probe: one real optimizer step on the longest BATCH*GA rows through the actual Trainer path
#    (fwd/bwd + gradient checkpointing + fp16 GradScaler + AdamW state allocation, which dominates peak VRAM).
#    If this OOMs, the real run will too -> lower BATCH_SIZE / raise GA.
# 3) 최악배치 프로브: 최장 행 BATCH*GA개로 실제 Trainer 경로에서 옵티마이저 1스텝. 여기서 OOM이면 본학습도 난다.
_idx = np.argsort(train_ds.lengths)[-(BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS):]
print('Probe batch lengths:', sorted(train_ds.lengths[i] for i in _idx))

class _Probe(Dataset):
    def __init__(self, base, idx): self.base, self.idx = base, list(idx)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i): return self.base[self.idx[i]]
    @property
    def lengths(self): return [self.base.lengths[i] for i in self.idx]

_snap = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}   # weight snapshot (~1.7 GB RAM)
_pa = TrainingArguments(                       # memory-path settings mirror the real run
    output_dir=str(OUT_DIR / '_probe'),
    max_steps=1,                               # GA micro-batches + a single optimizer step
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LR, weight_decay=WEIGHT_DECAY,
    fp16=True, bf16=False,                     # must match the real run — GradScaler is part of the memory path
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    save_strategy='no',                        # V3 left a 5GB probe checkpoint behind; nothing is written now
    dataloader_num_workers=2,
    report_to='none', disable_tqdm=False, remove_unused_columns=False,
    seed=SEED,
)
torch.cuda.reset_peak_memory_stats()
try:
    _pt = GroupedTrainer(model=model, args=_pa,
                         train_dataset=_Probe(train_ds, _idx), data_collator=collate_fn)
    _pt.train()
    print(f'Worst-batch probe passed — peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB')

    # 3b) Eval-path worst batch: the longest EVAL_BATCH_SIZE rows through `trainer` (eval batch = EVAL_BATCH_SIZE),
    #     measured while the optimizer state is still resident, as at the first real eval.
    # 3b) 평가 경로 최악 배치 — 옵티마이저 상태가 상주한 지금 재야 본학습 첫 eval 과 같은 조건이 된다.
    _idx_e = np.argsort(train_ds.lengths)[-EVAL_BATCH_SIZE:]
    torch.cuda.reset_peak_memory_stats()
    _ = trainer.predict(_Probe(train_ds, _idx_e))
    print(f'Eval-path probe passed  — peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB '
          f'(batch {EVAL_BATCH_SIZE}, lengths {min(train_ds.lengths[i] for i in _idx_e)}–{max(train_ds.lengths[i] for i in _idx_e)})')
finally:
    # Always restore weights and free the snapshot, even if the probe OOMs mid-step
    # 프로브가 도중에 죽더라도 가중치 복원과 스냅샷 해제는 반드시 수행한다
    model.load_state_dict(_snap)               # undo the probe's single optimizer step
    del _snap
    for _name in ['_pt', '_pa']:
        if _name in globals():
            del globals()[_name]
    for _ in range(3):
        gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(OUT_DIR / '_probe', ignore_errors=True)

print('All smoke tests passed — safe to proceed')

encoder loaded — 11 unused key(s) dropped (pretraining heads: ['lm_predictions', 'mask_predictions'])


Smoke tests 1 & 2 passed
Probe batch lengths: [2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048]


Step,Training Loss


Worst-batch probe passed — peak VRAM 26.1 GiB


Eval-path probe passed  — peak VRAM 33.1 GiB (batch 32, lengths 2048–2048)


All smoke tests passed — safe to proceed


In [23]:
mem_free()

memory cleared


# 7. Train / 학습

### Train / 학습
730스텝(¼ 에폭)마다 valid 5,000 으로 평가하고 저장한다. **고르는 규칙은 학습 전에 고정한다** — valid `eval_logloss`(정방향) 최소.
5번 연속 개선이 없으면 멈춘다(약 1.25 에폭). V3 곡선(에폭 1 → 2 → 3 : 0.9920 → 1.0193 → 1.2961)대로라면 2에폭 근처에서 끝난다.<br>
<br>
중단돼도 같은 명령으로 다시 돌리면 마지막 체크포인트에서 이어 간다. 학습이 끝난 뒤의 재실행은 `TRAINING_DONE.json` 을 보고 export 로 건너뛴다.

In [24]:
# Re-seed first — the smoke tests consumed RNG, and the real head's init must not depend on them.
# 스모크 테스트가 RNG 를 소비했으므로 본 모델(헤드 초기화) 전에 다시 고정한다.
set_seeds(SEED)
model = PrefModel(str(BACKBONE_DIR), vocab_size=len(tokenizer), new_token_ids=[BR_ID]).to(DEVICE)

trainer = GroupedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=valid_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[TrainingLogger(str(LOG_CSV)),
               BestSaver(k=KEEP_TOP_K, output_dir=str(OUT_DIR), metric='eval_logloss', greater_is_better=False),
               EarlyStoppingCallback(early_stopping_patience=ES_PATIENCE)],
)

# Id audit at the point of use — the rows handed to the Trainer, against the hardcoded TR once more.
# 사용 지점 id 감사 — Trainer 에 실제로 넘어가는 행을 하드코딩된 TR 값으로 한 번 더 검산한다.
TRAIN_IDS = np.asarray(trainer.train_dataset.ids)
SEEN_BY_TRAINER = (len(TRAIN_IDS), int(TRAIN_IDS.sum()))
assert not (set(TRAIN_IDS.tolist()) & _rec), 'a recovered row reached the Trainer!!'
assert not (set(TRAIN_IDS.tolist()) & (_sets['valid'] | _sets['holdout'])), 'a stage2 row reached the Trainer!!'
if QUICK:
    print(f'QUICK — the Trainer sees a {SEEN_BY_TRAINER[0]}-row slice; the 46,729-row audit runs in the full run')
else:
    assert SEEN_BY_TRAINER == EXPECT['train'][:2], f'the Trainer gets {SEEN_BY_TRAINER}, not TR {EXPECT["train"][:2]}!!'
    print(f'*** Trainer audit passed — {SEEN_BY_TRAINER[0]:,} rows, id sum {SEEN_BY_TRAINER[1]:,} = TR · '
          f'0 recovered rows · 0 stage2 rows ***')

Seed = 42


encoder loaded — 11 unused key(s) dropped (pretraining heads: ['lm_predictions', 'mask_predictions'])


*** Trainer audit passed — 46,729 rows, id sum 99,951,289,368,307 = TR · 0 recovered rows · 0 stage2 rows ***


In [25]:
# Resume-aware training: a finished run skips straight to the export, an interrupted one resumes from its last
# checkpoint (needs save_only_model=False). BestSaver and EarlyStopping restore their own state on resume.
# 끝난 런은 바로 export 로, 중단된 런은 마지막 체크포인트에서 이어서 — BestSaver·EarlyStopping 도 상태를 복구한다.
if DONE_FILE.exists():
    DONE = json.loads(DONE_FILE.read_text(encoding='utf-8'))
    print('Training already finished — skipping trainer.train()')
else:
    last_ckpt = get_last_checkpoint(str(RUN_DIR)) if RUN_DIR.is_dir() else None
    print('Resume from:', last_ckpt or 'none — training from scratch')
    _t0 = time.time()
    trainer.train(resume_from_checkpoint=last_ckpt)
    _st = trainer.state
    DONE = dict(best_step=int(re.sub(r'\D', '', os.path.basename(_st.best_model_checkpoint))),
                best_metric=float(_st.best_metric),
                global_step=int(_st.global_step), max_steps=int(_st.max_steps), epoch=round(float(_st.epoch), 4),
                stopped_early=bool(_st.global_step < _st.max_steps),
                train_seconds=round(time.time() - _t0), resumed_from=last_ckpt,
                finished_at=datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
    DONE_FILE.write_text(json.dumps(DONE, indent=2), encoding='utf-8')

BEST_STEP = DONE['best_step']
print(json.dumps(DONE, indent=2))

Resume from: none — training from scratch


Step,Training Loss,Validation Loss,Logloss,Accuracy
730,1.102800,1.078074,1.078074,0.395000
1460,1.052800,1.035859,1.035859,0.472200
2190,1.030000,1.018676,1.018676,0.479200
2920,1.024800,1.023010,1.023010,0.474400
3650,0.906900,1.025994,1.025994,0.499800
4380,0.949600,1.007960,1.007960,0.506200
5110,0.929500,1.012912,1.012912,0.502800
5840,0.931500,1.010746,1.010746,0.506400
6570,0.655100,1.206023,1.206023,0.486000
7300,0.746400,1.192353,1.192353,0.487200


[BestSaver] rank 1 (eval_logloss=1.0781) -> /workspace/abe1_out/full/best_checkpoints/checkpoint-730


[BestSaver] rank 1 (eval_logloss=1.0359) -> /workspace/abe1_out/full/best_checkpoints/checkpoint-1460


[BestSaver] rank 1 (eval_logloss=1.0187) -> /workspace/abe1_out/full/best_checkpoints/checkpoint-2190


...

[BestSaver] rank 2 (eval_logloss=1.0129) -> /workspace/abe1_out/full/best_checkpoints/checkpoint-5110


[BestSaver] removing 1.0230 @ step 2920


[BestSaver] rank 2 (eval_logloss=1.0107) -> /workspace/abe1_out/full/best_checkpoints/checkpoint-5840


[BestSaver] removing 1.0187 @ step 2190


{
  "best_step": 4380,
  "best_metric": 1.0079596261548864,
  "global_step": 8030,
  "max_steps": 8763,
  "epoch": 2.7491,
  "stopped_early": true,
  "train_seconds": 30827,
  "resumed_from": null,
  "finished_at": "2026-10-27 18:56:54"
}


# 8. Save & export / 저장과 로짓

In [26]:
# Save the architecture next to the weights. BestSaver copies weights only, and the Trainer keeps no config for a
# custom nn.Module — one config + the tokenizer go into the folder root (the ABE2 convention).
# BestSaver 사본은 가중치뿐이고 커스텀 nn.Module 이라 Trainer 가 config 를 남기지 않는다 — 루트에 1벌 둔다.
trainer.model.encoder.config.save_pretrained(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)

_best_w = BEST_DIR / f'checkpoint-{BEST_STEP}' / 'model.safetensors'
assert _best_w.exists(), f'BestSaver holds no copy of the trainer best (checkpoint-{BEST_STEP})!!'
_run_w = RUN_DIR / f'checkpoint-{BEST_STEP}' / 'model.safetensors'
BEST_SHA256 = sha256(_best_w)
if _run_w.exists():   # load_best_model_at_end keeps the best under run/ — the copy must be the same bytes
    assert BEST_SHA256 == sha256(_run_w), 'the BestSaver copy differs from the Trainer checkpoint!!'

print(f'trainer best  : checkpoint-{BEST_STEP}  (sha256 {BEST_SHA256[:16]}…)')
print(f'kept (top-{KEEP_TOP_K}): {sorted(int(p.name.split("-")[1]) for p in BEST_DIR.glob("checkpoint-*"))}')
print(f'root files    : {sorted(p.name for p in BEST_DIR.iterdir() if p.is_file())}')

trainer best  : checkpoint-4380  (sha256 e11ef2af475d90ac…)
kept (top-3): [4380, 5110, 5840]
root files    : ['added_tokens.json', 'best_k_state.json', 'config.json', 'special_tokens_map.json', 'spm.model', 'tokenizer.json', 'tokenizer_config.json']


### Export / 로짓 저장
남은 체크포인트(top-3)마다 valid·holdout 의 정방향·스왑 로짓을 `logits_export.npz` 로 남긴다(ABE2 와 같은 스키마).
다시 잰 valid 가 Trainer 기록과 2e-3 안에서 같아야 통과한다 — 가중치 로드가 어긋나면 여기서 잡힌다.
holdout 열은 보고용이다. 고르는 데 쓰지 않는다.

In [27]:
# Both views' logits for every kept checkpoint, on valid and holdout. A bi-encoder swap only exchanges a and b,
# so each branch is encoded once and the head runs twice — the same numbers as two Trainer.predict passes, half the cost.
# 남은 체크포인트마다 valid·holdout 의 두 뷰 로짓. 스왑은 a·b 교환뿐이라 브랜치당 인코딩 1회 + 헤드 2회로 충분하다.
@torch.no_grad()
def tta_logits(mdl, ds, batch=EVAL_BATCH_SIZE):
    mdl.eval()
    order = np.argsort(ds.lengths)[::-1]          # longest first — an OOM would show up in the first batch
    zf = np.zeros((len(ds), 3), np.float32)
    zs = np.zeros((len(ds), 3), np.float32)
    for s in tqdm(range(0, len(ds), batch), desc='tta', leave=False):
        idx = order[s:s + batch]
        bt = collate_fn([{'a_ids': ds.seq_a[i], 'b_ids': ds.seq_b[i]} for i in idx])
        # the same autocast region as PrefModel.forward: encoder, pooling and head
        # PrefModel.forward 와 같은 autocast 구간 — 인코더·풀링·헤드
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            a = mdl._encode(bt['a_ids'].to(DEVICE), bt['a_mask'].to(DEVICE))
            b = mdl._encode(bt['b_ids'].to(DEVICE), bt['b_mask'].to(DEVICE))
            f = mdl.head(torch.cat([a, b, a - b], dim=1))
            w = mdl.head(torch.cat([b, a, b - a], dim=1))
        zf[idx] = f.float().cpu().numpy()
        zs[idx] = w.float().cpu().numpy()[:, [1, 0, 2]]    # realign a/b logits / a·b 로짓 재정렬
    return zf, zs

ll = lambda z, y: log_loss(y, torch.softmax(torch.tensor(z, dtype=torch.float32), 1).numpy(), labels=[0, 1, 2])
META = dict(valid_ids=valid_df['id'].to_numpy(), valid_labels=np.array(valid_ds.labels),
            hold_ids=holdout_df['id'].to_numpy(),  hold_labels=np.array(holdout_ds.labels))
_logged = {e['step']: e['score'] for e in json.loads((BEST_DIR / 'best_k_state.json').read_text(encoding='utf-8'))}

trainer.optimizer = None                      # the optimizer state is not needed any more / 옵티마이저 상태 해제
gc.collect()
torch.cuda.empty_cache()

rows = []
for path in sorted(BEST_DIR.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1])):
    step = int(path.name.split('-')[1])
    out = path / 'logits_export.npz'
    if out.exists():                          # a rerun after a crash reuses what is already there
        with np.load(out) as z:
            vf, vs, hf, hs = z['valid_fwd'], z['valid_swp'], z['hold_fwd'], z['hold_swp']
    else:
        trainer.model.load_state_dict(load_file(path / 'model.safetensors'))   # strict — every key must match
        vf, vs = tta_logits(trainer.model, valid_ds)
        hf, hs = tta_logits(trainer.model, holdout_ds)
        assert all(np.isfinite(x).all() for x in (vf, vs, hf, hs)), f'checkpoint-{step}: non-finite logits!!'
        np.savez(out, valid_fwd=vf, valid_swp=vs, hold_fwd=hf, hold_swp=hs, **META)
    rows.append(dict(step=step, epoch=round(step / STEPS_PER_EPOCH, 2), pick='★' if step == BEST_STEP else '',
                     v_logged=_logged.get(step, np.nan), v_fwd=ll(vf, META['valid_labels']),
                     v_tta=ll((vf + vs) / 2, META['valid_labels']),
                     h_fwd=ll(hf, META['hold_labels']), h_tta=ll((hf + hs) / 2, META['hold_labels'])))
    print(f'  checkpoint-{step:<6} TTA log_loss  valid {rows[-1]["v_tta"]:.4f}  hold {rows[-1]["h_tta"]:.4f}')

df_ckpt = pd.DataFrame(rows)
df_ckpt['drift'] = (df_ckpt['v_fwd'] - df_ckpt['v_logged']).abs()
df_ckpt.to_csv(OUT_DIR / 'checkpoints.csv', index=False, encoding='utf-8')
display(df_ckpt.round(4))

# Re-measured valid (forward view) must reproduce what the Trainer logged — a weight-loading slip shows up here.
# 다시 잰 valid(정방향)가 Trainer 기록과 같아야 한다 — 가중치 로드가 어긋나면 여기서 드러난다.
assert (df_ckpt['drift'] < 2e-3).all(), 'a kept checkpoint does not reproduce its logged eval_logloss!!'
print('The pick was fixed before training: the ★ row has the lowest valid eval_logloss. '
      'The hold columns are a report, not a choice.')

tta:   0%|          | 0/157 [00:00<?, ?it/s]

tta:   0%|          | 0/180 [00:00<?, ?it/s]

  checkpoint-4380   TTA log_loss  valid 1.0069  hold 1.0046


tta:   0%|          | 0/157 [00:00<?, ?it/s]

tta:   0%|          | 0/180 [00:00<?, ?it/s]

...

tta:   0%|          | 0/157 [00:00<?, ?it/s]

tta:   0%|          | 0/180 [00:00<?, ?it/s]

  checkpoint-5840   TTA log_loss  valid 1.0101  hold 1.0036


,step,epoch,pick,v_logged,v_fwd,v_tta,h_fwd,h_tta,drift
0,4380,1.50,★,1.0080,1.0080,1.0069,1.0057,1.0046,0.0
1,5110,1.75,,1.0129,1.0129,1.0111,1.0064,1.0054,0.0
2,5840,2.00,,1.0107,1.0108,1.0101,1.0037,1.0036,0.0


The pick was fixed before training: the ★ row has the lowest valid eval_logloss. The hold columns are a report, not a choice.


# 9. Report / 보고

### Report / 보고
T 는 valid 로만 맞춘다(V3 규칙). 축 분해 항등식(tie + 승부 × 비-tie 비율 = 3-class)이 배선 점검을 겸한다.
valid 를 clean 2,000 / 회수 3,000 으로 쪼개 함께 찍는다 — 한쪽에만 이득이 있으면 선택 편향의 신호다.

In [28]:
# The pick, reported. T is fitted on valid only (the V3 rule) — the downstream pipeline never uses T.
# 고른 체크포인트의 보고. T 는 valid 로만 맞춘다(V3 규칙) — 하류 파이프라인은 T 를 쓰지 않는다.
with np.load(BEST_DIR / f'checkpoint-{BEST_STEP}' / 'logits_export.npz') as _z:
    Z = {k: _z[k] for k in _z.files}
v_avg = (Z['valid_fwd'] + Z['valid_swp']) / 2
h_avg = (Z['hold_fwd'] + Z['hold_swp']) / 2
vl, hl = Z['valid_labels'], Z['hold_labels']

# Temperature scaling: fit a single scalar T on the validation TTA logits
# T 스케일링: valid TTA 로짓으로 스칼라 T 하나만 피팅 (확신도만 보정)
def fit_temperature(logits, labels):
    z = torch.tensor(logits, dtype=torch.float32)
    y = torch.tensor(labels)
    logT = torch.zeros(1, requires_grad=True)      # log-param keeps T > 0
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = nn.functional.cross_entropy(z / logT.exp(), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(logT.exp().detach())

# binary logloss + AUC for one axis / 축 하나의 이진 logloss 와 AUC
def binary(score, lab):
    s = np.clip(score, 1e-12, 1 - 1e-12)
    ll_ = -(lab * np.log(s) + (1 - lab) * np.log(1 - s)).mean()
    o = np.argsort(s); rank = np.empty(len(s)); rank[o] = np.arange(1, len(s) + 1)
    n1 = lab.sum(); n0 = len(lab) - n1
    return ll_, (rank[lab.astype(bool)].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

T = fit_temperature(v_avg, vl)
P = torch.softmax(torch.tensor(h_avg, dtype=torch.float32), 1).numpy()
hold_tta   = log_loss(hl, P, labels=[0, 1, 2])
hold_tta_T = log_loss(hl, torch.softmax(torch.tensor(h_avg / T, dtype=torch.float32), 1).numpy(), labels=[0, 1, 2])

# total = tie axis + win axis x non-tie share — an identity, so it doubles as a wiring check
# 총손실 = tie 축 + 승부축 x 비-tie 비율 — 항등식이라 배선 점검을 겸한다
nt = hl != 2
ll_tie, auc_tie = binary(P[:, 2], (hl == 2).astype(float))
ll_win, auc_win = binary(P[nt, 0] / (P[nt, 0] + P[nt, 1]), (hl[nt] == 0).astype(float))
frac = nt.mean()
assert abs(ll_tie + ll_win * frac - hold_tta) < 1e-5, 'the axis decomposition does not add up — labels misaligned!!'

# valid split into ABE1(V3)'s own clean 2,000 and the 3,000 recovered rows — a gain on one half only would flag selection bias
# valid 를 clean 2,000 과 회수 3,000 으로 쪼갠다 — 한쪽에만 이득이 있으면 선택 편향 신호다
_is_rec = np.isin(Z['valid_ids'], recovered_df['id'].to_numpy())
V_HALVES = {nm: ll(v_avg[m], vl[m]) for nm, m in (('clean', ~_is_rec), ('recovered', _is_rec)) if m.any()}

REPORT = dict(T=T, valid_tta=ll(v_avg, vl), **{f'valid_tta_{k}': v for k, v in V_HALVES.items()},
              hold_fwd=ll(Z['hold_fwd'], hl), hold_tta=hold_tta, hold_tta_T=hold_tta_T,
              hold_tie_ll=ll_tie, hold_tie_auc=auc_tie, hold_win_ll=ll_win, hold_win_auc=auc_win,
              hold_acc=float((P.argmax(1) == hl).mean()))

print(f'checkpoint-{BEST_STEP} (epoch {BEST_STEP / STEPS_PER_EPOCH:.2f})   T = {T:.4f} (valid {len(vl):,})')
print(f'  valid TTA        {REPORT["valid_tta"]:.4f}   ' +
      '  '.join(f'{k} {v:.4f}' for k, v in V_HALVES.items()))
print(f'  hold fwd         {REPORT["hold_fwd"]:.4f}')
print(f'  hold TTA         {hold_tta:.4f}   acc {REPORT["hold_acc"] * 100:.2f}%')
print(f'  hold TTA + T     {hold_tta_T:.4f}')
print(f'  tie axis         ll {ll_tie:.4f}  AUC {auc_tie:.4f}   (n={len(hl):,})')
print(f'  win axis         ll {ll_win:.4f}  AUC {auc_win:.4f}   (n={nt.sum():,})')
print(f'  verify           {ll_tie:.4f} + {ll_win:.4f} x {frac:.4f} = {ll_tie + ll_win * frac:.6f}  vs  {hold_tta:.6f}')

# Earlier bi-encoders on the same 5,748 holdout (TTA) / 같은 holdout 5,748 에서 잰 이전 bi-encoder 들
REF = {'V3 ABE1 ckpt-3109 (49,729 rows, the 3,000 included)': 0.9888,
       'ABE2 ckpt-2500 (RM backbone, 4-block head)':           0.9928,
       'V4 ckpt-5250 (base backbone, 4-block head, 5e-6)':     1.0016}
print('\nreference (hold TTA)')
for _k, _v in REF.items():
    print(f'  {_k:<55} {_v:.4f}   this run {hold_tta - _v:+.4f}')
if not QUICK and hold_tta > 1.000:
    print('\n★ WARN — hold TTA above 1.000, worse than both V3 and ABE2. '
          'Read DECISIONS.md "If it comes back worse" before integrating.')

checkpoint-4380 (epoch 1.50)   T = 1.3197 (valid 5,000)
  valid TTA        1.0069   clean 1.0017  recovered 1.0103
  hold fwd         1.0057
  hold TTA         1.0046   acc 50.84%
  hold TTA + T     0.9960
  tie axis         ll 0.5870  AUC 0.6386   (n=5,748)
  win axis         ll 0.6043  AUC 0.7477   (n=3,972)
  verify           0.5870 + 0.6043 x 0.6910 = 1.004581  vs  1.004581

reference (hold TTA)
  V3 ABE1 ckpt-3109 (49,729 rows, the 3,000 included)     0.9888   this run +0.0158
  ABE2 ckpt-2500 (RM backbone, 4-block head)              0.9928   this run +0.0118
  V4 ckpt-5250 (base backbone, 4-block head, 5e-6)        1.0016   this run +0.0030

★ WARN — hold TTA above 1.000, worse than both V3 and ABE2. Read DECISIONS.md "If it comes back worse" before integrating.


# 10. Deliver / 회수물

### Deliver / 회수물
`deliver/ABE1/` 은 최종 노트북이 읽는 `bundle/models/ABE1` 과 같은 모양이다(config + 토크나이저 + `checkpoint-<step>/model.safetensors`).

In [29]:
# What comes home: ABE1/ shaped like bundle/models/ABE1, plus the records.
# 회수물: bundle/models/ABE1 과 같은 모양의 ABE1/ + 기록.
if DELIVER_DIR.exists():
    shutil.rmtree(DELIVER_DIR)
_abe = DELIVER_DIR / 'ABE1'
_ck = _abe / f'checkpoint-{BEST_STEP}'
_ck.mkdir(parents=True)
for _f in ('model.safetensors', 'logits_export.npz'):
    shutil.copy2(BEST_DIR / f'checkpoint-{BEST_STEP}' / _f, _ck / _f)
shutil.copy2(BEST_DIR / 'config.json', _abe / 'config.json')
tokenizer.save_pretrained(_abe)

# the runner-up checkpoints come home as logits only; their weights stay on the volume
# 나머지 체크포인트는 로짓만 가져온다 — 가중치는 볼륨에 남는다
for _p in sorted(BEST_DIR.glob('checkpoint-*')):
    if _p.name != f'checkpoint-{BEST_STEP}':
        (DELIVER_DIR / 'others' / _p.name).mkdir(parents=True)
        shutil.copy2(_p / 'logits_export.npz', DELIVER_DIR / 'others' / _p.name / 'logits_export.npz')

_r = DELIVER_DIR / 'records'
_r.mkdir()
for _src in (LOG_CSV, BEST_DIR / 'best_k_state.json', OUT_DIR / 'checkpoints.csv', DONE_FILE):
    shutil.copy2(_src, _r / _src.name)
_last = get_last_checkpoint(str(RUN_DIR)) if RUN_DIR.is_dir() else None
if _last:
    shutil.copy2(Path(_last) / 'trainer_state.json', _r / 'trainer_state.json')    # the full log_history

_py = lambda o: o.item() if hasattr(o, 'item') else str(o)    # numpy scalars -> json
SUMMARY = dict(
    kind=KIND,
    best_step=BEST_STEP,
    selection_rule=(f'lowest eval_logloss on valid {len(valid_df):,} (forward view), '
                    f'evaluated every {EVAL_STEPS} steps — fixed before training'),
    done=DONE,
    metrics=REPORT,
    checkpoints=df_ckpt.to_dict(orient='records'),
    ids=dict(expected={k: list(v[:2]) for k, v in EXPECT.items()},
             seen_by_trainer=list(SEEN_BY_TRAINER),
             recovered_rows_seen_by_trainer=0),
    model_sha256=BEST_SHA256,
    hparams=dict(lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, grad_accum=GRADIENT_ACCUMULATION_STEPS,
                 warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY, max_len=MAX_LEN, q_floor=Q_FLOOR,
                 head_frac=HEAD_FRAC, eval_steps=EVAL_STEPS, es_patience=ES_PATIENCE, keep_top_k=KEEP_TOP_K,
                 eval_batch_size=EVAL_BATCH_SIZE, fp16=True, seed=SEED),
    env=dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda,
             transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0), vram_gb=round(VRAM_GB, 1)),
    written_at=datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
)
# explicit utf-8: a pod's default text encoding can be ASCII, and the summary carries '—' and '★'
# utf-8 명시 — 파드의 기본 텍스트 인코딩이 ASCII 일 수 있다 (2026-10-27 A40 QUICK 에서 실제로 막혔다)
(DELIVER_DIR / 'run_summary.json').write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False, default=_py),
                                              encoding='utf-8')

_total = 0
for _p in sorted(DELIVER_DIR.rglob('*')):
    if _p.is_file():
        _total += _p.stat().st_size
        print(f'  {str(_p.relative_to(DELIVER_DIR)):<60} {_p.stat().st_size / 2**20:>9.1f} MB')
print(f'deliver/ {_total / 2**30:.2f} GiB — run.sh adds the executed notebook and tars it')

  ABE1/added_tokens.json                                             0.0 MB
  ABE1/checkpoint-4380/logits_export.npz                             0.4 MB
  ABE1/checkpoint-4380/model.safetensors                          1667.7 MB
  ABE1/config.json                                                   0.0 MB
  ABE1/special_tokens_map.json                                       0.0 MB
  ABE1/spm.model                                                     2.4 MB
  ABE1/tokenizer.json                                                8.3 MB
  ABE1/tokenizer_config.json                                         0.0 MB
  others/checkpoint-5110/logits_export.npz                           0.4 MB
  others/checkpoint-5840/logits_export.npz                           0.4 MB
  records/TRAINING_DONE.json                                         0.0 MB
  records/best_k_state.json                                          0.0 MB
  records/checkpoints.csv                                            0.0 MB
  records/tr

In [30]:
print(f'*** Done — {KIND} run · checkpoint-{BEST_STEP} · {DELIVER_DIR} ***')

*** Done — FULL run · checkpoint-4380 · /workspace/abe1_out/full/deliver ***


# 끝.